# ACE Studio — ACE-Step 1.5 · Kaggle T4×2 · Cloudflare Tunnel

这份 Notebook 把原来的实验式推理流程重构成一个完整的 **AI Music Studio**：

```text
Browser
  │
  │ HTTPS · TryCloudflare
  ▼
Cloudflare Quick Tunnel
  │
  ▼
FastAPI :7860
  ├── Custom Studio UI
  ├── Job / History API
  ├── WAV Streaming / Download
  └── Scheduler
       ├── T4 #0 · persistent ACE-Step worker
       └── T4 #1 · persistent ACE-Step worker
```

### 设计目标

- **完整 UI/UX**：不是 Gradio 默认界面，而是定制“录音棚控制台 + 黑胶 / 频谱”工作台。
- **双 T4 常驻**：每张 T4 各加载一次 `acestep-v15-turbo`，后续任务排队复用热模型。
- **T4 稳定路径**：默认 `DiT-only + instrumental + thinking=False + 8 steps`。
- **环境隔离**：ACE-Step 子进程不继承 Kaggle 的 `PYTHONPATH / MPLBACKEND`，避免之前的 `wrapt / matplotlib_inline` 报错。
- **同源前后端**：前端和 API 都由同一个 FastAPI 服务提供，不需要 CORS。
- **CF Tunnel**：自动创建 `trycloudflare.com` HTTPS 地址。
- **无密码访问**：Cloudflare Tunnel 地址打开即用，不需要 authentication / 登录 / URL 参数。
- **轮询状态**：不依赖 SSE/WebSocket，适配 Cloudflare Quick Tunnel。

> 注意：这是**无认证公网入口**。任何拿到当前 `trycloudflare.com` URL 的人都可以调用生成接口，占用本次 Kaggle GPU。

> Kaggle Notebook Settings：选择 **GPU T4 x2**，并打开 **Internet**。

In [1]:
# Cell 1 — 环境、仓库、独立 venv
#
# 这一格完成：
# 1) 检查 T4 x2
# 2) 拉取/更新 ACE-Step 1.5
# 3) uv sync
# 4) 给 Studio 后端补充 FastAPI / Uvicorn
# 5) 定义“干净子进程环境”，彻底隔离 Kaggle sitecustomize / matplotlib_inline

from pathlib import Path
import os
import sys
import subprocess

WORK = Path("/kaggle/working")
PROJECT = WORK / "ACE-Step-1.5"

print("=== GPU ===")
subprocess.run(["nvidia-smi"], check=False)

try:
    import torch
    print("\nKernel torch:", torch.__version__)
    print("GPU count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(
            f"GPU {i}: {p.name} | "
            f"{p.total_memory / 1024**3:.1f} GiB | CC {p.major}.{p.minor}"
        )
    assert torch.cuda.device_count() >= 2, (
        "没有检测到 T4 x2。请在 Kaggle Notebook Settings 中选择 GPU T4 x2。"
    )
except Exception as e:
    print("Kernel torch 检查：", repr(e))
    print("ACE-Step 后续仍使用自己的 .venv；但请确认 Kaggle 已启用 T4 x2。")


def clean_subprocess_env(extra=None):
    env = os.environ.copy()
    env.pop("PYTHONPATH", None)
    env.pop("PYTHONHOME", None)
    env["PYTHONNOUSERSITE"] = "1"
    env["MPLBACKEND"] = "Agg"
    env["MPLCONFIGDIR"] = "/tmp/ace_matplotlib"
    env["PYTHONUNBUFFERED"] = "1"
    if extra:
        env.update({str(k): str(v) for k, v in extra.items()})
    return env


# 安装 uv（只作为环境管理器）
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "uv"],
    check=True,
)

if not PROJECT.exists():
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/ace-step/ACE-Step-1.5.git",
            str(PROJECT),
        ],
        check=True,
    )
else:
    print("\n复用现有仓库并更新到当前 origin/main")
    subprocess.run(["git", "fetch", "--depth", "1", "origin", "main"], cwd=PROJECT, check=True)
    subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=PROJECT, check=True)

print("\n=== uv sync ===")
subprocess.run(
    ["uv", "sync"],
    cwd=PROJECT,
    env=clean_subprocess_env(),
    check=True,
)

ACE_PY = PROJECT / ".venv" / "bin" / "python"
assert ACE_PY.exists(), f"ACE-Step venv 不存在: {ACE_PY}"

# Studio Web 服务依赖放进 ACE-Step 自己的 venv。
subprocess.run(
    [
        "uv", "pip", "install",
        "--python", str(ACE_PY),
        "fastapi>=0.115",
        "uvicorn>=0.30",
    ],
    cwd=PROJECT,
    env=clean_subprocess_env(),
    check=True,
)

commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"],
    cwd=PROJECT,
    text=True,
).strip()

probe = r"""
import os, sys
import matplotlib
import fastapi, uvicorn
print("python      =", sys.executable)
print("MPLBACKEND  =", os.environ.get("MPLBACKEND"))
print("backend     =", matplotlib.get_backend())
print("fastapi     =", fastapi.__version__)
print("uvicorn     =", uvicorn.__version__)
"""

print("\n=== isolated venv probe ===")
subprocess.run(
    [str(ACE_PY), "-c", probe],
    cwd=PROJECT,
    env=clean_subprocess_env(),
    check=True,
)

print("\nACE-Step commit:", commit)
print("Project:", PROJECT)

=== GPU ===
Thu Sep  3 08:07:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------

Cloning into '/kaggle/working/ACE-Step-1.5'...



=== uv sync ===


Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 176 packages in 25.51s
Prepared 141 packages in 43.86s
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 141 packages in 19.57s
 + absl-py==2.4.0
 + accelerate==1.12.0
 + ace-step==1.5.0 (from file:///kaggle/working/ACE-Step-1.5)
 + aiofiles==24.1.0
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.3
 + aiosignal==1.4.0
 + annotated-doc==0.0.4
 + annotated-types==0.7.0
 + anyio==4.12.1
 + attrs==25.4.0
 + brotli==1.2.0
 + certifi==2026.1.4
 + cffi==2.0.0
 + charset-normalizer==3.4.4
 + click==8.3.1
 + contourpy==1.3.3
 + cuda-bindings==12.9.4
 + cuda-pathfinder==1.3.3
 + cycler==0.12.1
 + diffusers==0.37.1
 + diskcache==5.6.3
 + einops==0.8.2
 + einx==0.3.0
 + fastapi==0.128.0
 + ffmpy==1.0.0
 + filelock==3.2


=== isolated venv probe ===
python      = /kaggle/working/ACE-Step-1.5/.venv/bin/python
MPLBACKEND  = Agg
backend     = Agg
fastapi     = 0.128.0
uvicorn     = 0.40.0

ACE-Step commit: ca1e85f
Project: /kaggle/working/ACE-Step-1.5


In [2]:
# Cell 2 — Hugging Face Token（可选）+ 官方模型下载
#
# 如果你在 Kaggle Secrets 里配置了 HF_TOKEN，会自动使用。
# 没有也可以下载公开权重。
#
# 注意：运行时 Studio 使用 Turbo DiT，不加载 5Hz LM；
# 这里仍采用 ACE-Step 官方 downloader，优先保证 checkpoints 结构完整。

from pathlib import Path
import os
import subprocess

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGING_FACE_HUB_TOKEN"] = token
        print("HF_TOKEN: loaded")
    else:
        print("HF_TOKEN: not set — anonymous download")
except Exception:
    print("HF_TOKEN: not set — anonymous download")

env = clean_subprocess_env()
if os.environ.get("HF_TOKEN"):
    env["HF_TOKEN"] = os.environ["HF_TOKEN"]
    env["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]

print("\n=== ACE-Step checkpoints ===")
subprocess.run(
    [str(ACE_PY), "-m", "acestep.model_downloader"],
    cwd=PROJECT,
    env=env,
    check=True,
)

checkpoint_dir = PROJECT / "checkpoints"
for p in sorted(checkpoint_dir.glob("*")):
    print("✓", p.name)

HF_TOKEN: not set — anonymous download

=== ACE-Step checkpoints ===
Checkpoints directory: /kaggle/working/ACE-Step-1.5/checkpoints
Destination: /kaggle/working/ACE-Step-1.5/checkpoints
This may take a while depending on your internet connection...


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'
2026-09-03 08:09:32.683 | INFO     | __main__:_smart_download:249 - [Model Download] Auto-detected: HuggingFace Hub
2026-09-03 08:09:32.683 | INFO     | __main__:_smart_download:252 - [Model Download] Using HuggingFace Hub...
2026-09-03 08:09:33.166 | INFO     | __main__:_download_from_huggingface_internal:180 - [Model Download] Downloading from HuggingFace: ACE-Step/Ace-Step1.5 -> /kaggle/working/ACE-Step-1.5/checkpoints
Fetching 28 files:  21%|██▏       | 6/28 [00:29<01:52,  5.13s/it]

Successfully downloaded from HuggingFace: ACE-Step/Ace-Step1.5

Download complete!
Models are available at: /kaggle/working/ACE-Step-1.5/checkpoints
✓ .cache
✓ .gitattributes
✓ Qwen3-Embedding-0.6B
✓ README.md
✓ acestep-5Hz-lm-1.7B
✓ acestep-v15-turbo
✓ config.json
✓ vae


Fetching 28 files: 100%|██████████| 28/28 [00:29<00:00,  1.06s/it]
2026-09-03 08:10:02.990 | DEBUG    | __main__:_sync_model_code_files:131 - [Model Sync] Synced configuration_acestep_v15.py -> /kaggle/working/ACE-Step-1.5/checkpoints/acestep-v15-turbo/configuration_acestep_v15.py
2026-09-03 08:10:02.992 | DEBUG    | __main__:_sync_model_code_files:131 - [Model Sync] Synced modeling_acestep_v15_turbo.py -> /kaggle/working/ACE-Step-1.5/checkpoints/acestep-v15-turbo/modeling_acestep_v15_turbo.py
2026-09-03 08:10:02.992 | INFO     | __main__:download_main_model:489 - [Model Download] Synced code files for acestep-v15-turbo: ['configuration_acestep_v15.py', 'modeling_acestep_v15_turbo.py']


In [3]:
# Cell 3 — 写入 ACE Studio 后端 + 定制 UI
#
# 后端：FastAPI + 双 GPU persistent workers + queue/history/audio API（无认证）
# 前端：完全自定义 HTML/CSS/JS，不依赖 Gradio，不依赖前端构建步骤。

from pathlib import Path

PROJECT = Path("/kaggle/working/ACE-Step-1.5")

SERVER_FILE = PROJECT / "ace_studio_server.py"
UI_FILE = PROJECT / "ace_studio_ui.html"

server_source = r"""from __future__ import annotations

import asyncio
import atexit
import gc
import json
import multiprocessing as mp
import os
import random
import shutil
import signal
import subprocess
import time
import traceback
import uuid
from contextlib import asynccontextmanager
from pathlib import Path
from typing import Any

from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse, HTMLResponse, JSONResponse
from pydantic import BaseModel, Field

PROJECT = Path(__file__).resolve().parent
UI_PATH = PROJECT / "ace_studio_ui.html"
OUTPUT_ROOT = Path(os.environ.get("ACE_STUDIO_OUTPUTS", "/kaggle/working/ace_studio_outputs"))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_CONFIG = os.environ.get("ACE_STUDIO_MODEL", "acestep-v15-turbo")
GPU_IDS = [int(x.strip()) for x in os.environ.get("ACE_STUDIO_GPUS", "0,1").split(",") if x.strip()]
STARTED_AT = time.time()

CTX = mp.get_context("spawn")
MANAGER = None
JOBS = None
GPU_STATES = None
QUEUE_COUNTS = None
TASK_QUEUES: dict[int, Any] = {}
WORKERS: dict[int, mp.Process] = {}
TELEMETRY_CACHE: dict[str, Any] = {"ts": 0.0, "rows": []}


class GenerateRequest(BaseModel):
    prompt: str = Field(min_length=3, max_length=700)
    duration: float = Field(default=30, ge=10, le=120)
    bpm: int = Field(default=82, ge=30, le=300)
    keyscale: str = Field(default="A minor", max_length=32)
    timesignature: str = Field(default="4", pattern=r"^(2|3|4|6)$")
    seed: int | None = Field(default=None, ge=0, le=2_147_483_647)
    steps: int = Field(default=8, ge=4, le=20)


def _now() -> float:
    return time.time()


def _job_update(jobs, job_id: str, **changes):
    current = dict(jobs.get(job_id, {}))
    current.update(changes)
    current["updated_at"] = _now()
    jobs[job_id] = current


def _gpu_update(gpu_states, gpu_id: int, **changes):
    key = str(gpu_id)
    current = dict(gpu_states.get(key, {}))
    current.update(changes)
    current["updated_at"] = _now()
    gpu_states[key] = current


def worker_main(gpu_id: int, task_queue, jobs, gpu_states, queue_counts, project: str, output_root: str):
    os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    os.environ["MPLBACKEND"] = "Agg"
    os.environ["MPLCONFIGDIR"] = f"/tmp/ace_matplotlib_gpu{gpu_id}"
    os.environ["PYTHONNOUSERSITE"] = "1"

    project_path = Path(project)
    output_root_path = Path(output_root)

    try:
        import torch

        prop = torch.cuda.get_device_properties(0)
        _gpu_update(
            gpu_states,
            gpu_id,
            state="loading",
            name=prop.name,
            memory_total_gb=round(prop.total_memory / 1024**3, 1),
            compute_capability=f"{prop.major}.{prop.minor}",
            detail="Loading ACE-Step Turbo…",
        )

        from acestep.handler import AceStepHandler
        from acestep.inference import GenerationConfig, GenerationParams, generate_music

        dit_handler = AceStepHandler()
        status, ok = dit_handler.initialize_service(
            project_root=str(project_path),
            config_path=MODEL_CONFIG,
            device="cuda",
            offload_to_cpu=False,
        )
        if not ok:
            raise RuntimeError(f"DiT init failed: {status}")

        _gpu_update(
            gpu_states,
            gpu_id,
            state="ready",
            detail="Turbo model ready",
            model=MODEL_CONFIG,
        )
    except Exception as exc:
        _gpu_update(
            gpu_states,
            gpu_id,
            state="error",
            detail=str(exc)[:1000],
            traceback=traceback.format_exc()[-5000:],
        )
        return

    while True:
        task = task_queue.get()
        if task is None:
            break

        job_id = task["id"]
        queue_counts[str(gpu_id)] = max(0, int(queue_counts.get(str(gpu_id), 1)) - 1)

        job = dict(jobs.get(job_id, {}))
        if job.get("status") == "canceled":
            continue

        _job_update(
            jobs,
            job_id,
            status="running",
            gpu=gpu_id,
            progress=0.08,
            stage="Preparing model inputs",
            started_at=_now(),
        )
        _gpu_update(
            gpu_states,
            gpu_id,
            state="busy",
            detail=f"Rendering {job_id[:8]}",
            current_job=job_id,
        )

        try:
            payload = task["payload"]
            job_dir = output_root_path / job_id
            job_dir.mkdir(parents=True, exist_ok=True)

            def progress_cb(value, desc=None, **_kwargs):
                try:
                    p = float(value)
                except Exception:
                    p = 0.5
                p = max(0.08, min(0.96, p))
                _job_update(
                    jobs,
                    job_id,
                    progress=p,
                    stage=str(desc or "Rendering audio"),
                )

            params = GenerationParams(
                task_type="text2music",
                thinking=False,
                caption=payload["prompt"],
                lyrics="[Instrumental]",
                instrumental=True,
                bpm=payload["bpm"],
                keyscale=payload["keyscale"],
                timesignature=payload["timesignature"],
                vocal_language="unknown",
                duration=payload["duration"],
                inference_steps=payload["steps"],
                seed=payload["seed"],
            )

            config = GenerationConfig(
                batch_size=1,
                seeds=[payload["seed"]],
                use_random_seed=False,
                audio_format="wav",
            )

            t0 = time.time()
            result = generate_music(
                dit_handler=dit_handler,
                llm_handler=None,
                params=params,
                config=config,
                save_dir=str(job_dir),
                progress=progress_cb,
            )

            if not result.success:
                raise RuntimeError(result.error or result.status_message or "ACE-Step generation failed")

            src_audio = None
            for item in result.audios:
                candidate = item.get("path") if isinstance(item, dict) else None
                if candidate and Path(candidate).exists():
                    src_audio = Path(candidate)
                    break

            if src_audio is None:
                wavs = sorted(job_dir.rglob("*.wav"), key=lambda p: p.stat().st_mtime, reverse=True)
                if wavs:
                    src_audio = wavs[0]

            if src_audio is None:
                raise FileNotFoundError("Generation succeeded but no WAV file was found")

            final_audio = job_dir / "render.wav"
            if src_audio.resolve() != final_audio.resolve():
                shutil.copy2(src_audio, final_audio)

            elapsed = time.time() - t0
            completed = dict(jobs.get(job_id, {}))
            completed.update(
                status="done",
                progress=1.0,
                stage="Ready",
                audio_path=str(final_audio),
                elapsed_seconds=round(elapsed, 2),
                completed_at=_now(),
                updated_at=_now(),
            )
            jobs[job_id] = completed

            meta = {k: v for k, v in completed.items() if k != "error"}
            (job_dir / "meta.json").write_text(
                json.dumps(meta, ensure_ascii=False, indent=2),
                encoding="utf-8",
            )

        except Exception as exc:
            _job_update(
                jobs,
                job_id,
                status="error",
                progress=0.0,
                stage="Failed",
                error=str(exc)[:1800],
                traceback=traceback.format_exc()[-5000:],
            )
            try:
                import torch
                torch.cuda.empty_cache()
            except Exception:
                pass
            gc.collect()
        finally:
            _gpu_update(
                gpu_states,
                gpu_id,
                state="ready",
                detail="Turbo model ready",
                current_job=None,
            )


def load_persisted_history():
    if JOBS is None:
        return
    for meta_path in OUTPUT_ROOT.glob("*/meta.json"):
        try:
            data = json.loads(meta_path.read_text(encoding="utf-8"))
            if data.get("id"):
                JOBS[data["id"]] = data
        except Exception:
            continue


def start_workers():
    global MANAGER, JOBS, GPU_STATES, QUEUE_COUNTS, TASK_QUEUES, WORKERS

    MANAGER = CTX.Manager()
    JOBS = MANAGER.dict()
    GPU_STATES = MANAGER.dict()
    QUEUE_COUNTS = MANAGER.dict()

    load_persisted_history()

    for gpu_id in GPU_IDS:
        GPU_STATES[str(gpu_id)] = {
            "id": gpu_id,
            "state": "booting",
            "detail": "Worker process starting…",
            "updated_at": _now(),
        }
        QUEUE_COUNTS[str(gpu_id)] = 0

        q = CTX.Queue()
        TASK_QUEUES[gpu_id] = q

        p = CTX.Process(
            target=worker_main,
            args=(gpu_id, q, JOBS, GPU_STATES, QUEUE_COUNTS, str(PROJECT), str(OUTPUT_ROOT)),
            name=f"ace-gpu-{gpu_id}",
            daemon=True,
        )
        p.start()
        WORKERS[gpu_id] = p


def stop_workers():
    global MANAGER
    for q in TASK_QUEUES.values():
        try:
            q.put_nowait(None)
        except Exception:
            pass

    for p in WORKERS.values():
        try:
            p.join(timeout=5)
            if p.is_alive():
                p.terminate()
        except Exception:
            pass

    if MANAGER is not None:
        try:
            MANAGER.shutdown()
        except Exception:
            pass
        MANAGER = None


def choose_gpu() -> int:
    if not GPU_IDS:
        raise HTTPException(status_code=503, detail="No GPU workers configured")

    candidates = []
    for gpu_id in GPU_IDS:
        state = dict(GPU_STATES.get(str(gpu_id), {}))
        if state.get("state") == "error":
            continue
        qn = int(QUEUE_COUNTS.get(str(gpu_id), 0))
        busy_penalty = 1 if state.get("state") == "busy" else 0
        candidates.append((qn + busy_penalty, qn, gpu_id))

    if not candidates:
        raise HTTPException(status_code=503, detail="All GPU workers are unavailable")

    candidates.sort()
    return candidates[0][2]


def gpu_telemetry():
    global TELEMETRY_CACHE
    now = time.time()
    if now - TELEMETRY_CACHE["ts"] < 1.2:
        return TELEMETRY_CACHE["rows"]

    rows = []
    try:
        out = subprocess.check_output(
            [
                "nvidia-smi",
                "--query-gpu=index,name,memory.used,memory.total,utilization.gpu,temperature.gpu",
                "--format=csv,noheader,nounits",
            ],
            text=True,
            timeout=3,
        )
        for line in out.strip().splitlines():
            parts = [x.strip() for x in line.split(",")]
            if len(parts) >= 6:
                rows.append(
                    {
                        "id": int(parts[0]),
                        "name": parts[1],
                        "memory_used_mb": int(parts[2]),
                        "memory_total_mb": int(parts[3]),
                        "utilization": int(parts[4]),
                        "temperature": int(parts[5]),
                    }
                )
    except Exception:
        rows = []

    TELEMETRY_CACHE = {"ts": now, "rows": rows}
    return rows



def public_job(job: dict[str, Any]) -> dict[str, Any]:
    safe = {k: v for k, v in job.items() if k not in {"audio_path", "traceback"}}
    if job.get("status") == "done":
        jid = job["id"]
        safe["audio_url"] = f"/api/audio/{jid}"
        safe["download_url"] = f"/api/download/{jid}"
    return safe


@asynccontextmanager
async def lifespan(_app: FastAPI):
    start_workers()
    yield
    stop_workers()


app = FastAPI(title="ACE Studio", version="1.0", lifespan=lifespan)


@app.get("/", response_class=HTMLResponse)
async def home():
    if not UI_PATH.exists():
        return HTMLResponse("<h1>ACE Studio UI file missing</h1>", status_code=500)
    return HTMLResponse(UI_PATH.read_text(encoding="utf-8"))


@app.get("/favicon.ico")
async def favicon():
    return JSONResponse(content={}, status_code=204)


@app.get("/api/status")
async def status():
    telemetry = {row["id"]: row for row in gpu_telemetry()}
    gpus = []
    for gpu_id in GPU_IDS:
        state = dict(GPU_STATES.get(str(gpu_id), {}))
        state["queue"] = int(QUEUE_COUNTS.get(str(gpu_id), 0))
        state["telemetry"] = telemetry.get(gpu_id)
        gpus.append(state)

    jobs = [dict(v) for v in JOBS.values()]
    active = sum(1 for j in jobs if j.get("status") in {"queued", "running"})
    done = sum(1 for j in jobs if j.get("status") == "done")

    return {
        "ok": True,
        "model": MODEL_CONFIG,
        "mode": "DiT-only · instrumental · T4 stable",
        "gpus": gpus,
        "jobs_active": active,
        "jobs_done": done,
        "uptime_seconds": round(time.time() - STARTED_AT),
    }


@app.post("/api/generate")
async def generate(req: GenerateRequest):
    gpu_id = choose_gpu()
    seed = req.seed if req.seed is not None else random.randint(1, 2_147_483_647)
    job_id = uuid.uuid4().hex

    payload = req.model_dump()
    payload["seed"] = seed

    job = {
        "id": job_id,
        "status": "queued",
        "progress": 0.03,
        "stage": "Queued",
        "gpu": gpu_id,
        "created_at": _now(),
        "updated_at": _now(),
        **payload,
    }
    JOBS[job_id] = job

    QUEUE_COUNTS[str(gpu_id)] = int(QUEUE_COUNTS.get(str(gpu_id), 0)) + 1
    TASK_QUEUES[gpu_id].put({"id": job_id, "payload": payload})

    return public_job(job)


@app.get("/api/jobs/{job_id}")
async def get_job(job_id: str):
    job = JOBS.get(job_id)
    if not job:
        raise HTTPException(status_code=404, detail="Job not found")
    return public_job(dict(job))


@app.get("/api/history")
async def history(limit: int = Query(default=24, ge=1, le=100)):
    jobs = [dict(v) for v in JOBS.values()]
    jobs.sort(key=lambda x: float(x.get("created_at", 0)), reverse=True)
    return [public_job(j) for j in jobs[:limit]]


@app.post("/api/jobs/{job_id}/cancel")
async def cancel_job(job_id: str):
    job = JOBS.get(job_id)
    if not job:
        raise HTTPException(status_code=404, detail="Job not found")
    data = dict(job)
    if data.get("status") != "queued":
        raise HTTPException(status_code=409, detail="Only queued jobs can be canceled")
    _job_update(JOBS, job_id, status="canceled", progress=0, stage="Canceled")
    return {"ok": True}


@app.delete("/api/jobs/{job_id}")
async def delete_job(job_id: str):
    job = JOBS.get(job_id)
    if not job:
        raise HTTPException(status_code=404, detail="Job not found")
    data = dict(job)
    if data.get("status") in {"queued", "running"}:
        raise HTTPException(status_code=409, detail="Active jobs cannot be deleted")
    job_dir = OUTPUT_ROOT / job_id
    if job_dir.exists():
        shutil.rmtree(job_dir, ignore_errors=True)
    try:
        del JOBS[job_id]
    except Exception:
        pass
    return {"ok": True}


@app.get("/api/audio/{job_id}")
async def audio(job_id: str):
    job = JOBS.get(job_id)
    if not job:
        raise HTTPException(status_code=404, detail="Job not found")
    path = Path(dict(job).get("audio_path", ""))
    if not path.exists():
        raise HTTPException(status_code=404, detail="Audio file not found")
    return FileResponse(path, media_type="audio/wav", filename=f"ace-{job_id[:8]}.wav")


@app.get("/api/download/{job_id}")
async def download(job_id: str):
    job = JOBS.get(job_id)
    if not job:
        raise HTTPException(status_code=404, detail="Job not found")
    path = Path(dict(job).get("audio_path", ""))
    if not path.exists():
        raise HTTPException(status_code=404, detail="Audio file not found")
    return FileResponse(
        path,
        media_type="audio/wav",
        filename=f"ace-studio-{job_id[:8]}.wav",
        content_disposition_type="attachment",
    )


if __name__ == "__main__":
    import uvicorn

    port = int(os.environ.get("ACE_STUDIO_PORT", "7860"))
    uvicorn.run(app, host="0.0.0.0", port=port, log_level="info")
"""

ui_source = r"""<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1" />
  <meta name="theme-color" content="#0b0b0a" />
  <title>ACE Studio — Generative Music Console</title>
  <link rel="preconnect" href="https://fonts.googleapis.com">
  <link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
  <link href="https://fonts.googleapis.com/css2?family=Instrument+Serif:ital@0;1&family=IBM+Plex+Mono:wght@400;500;600&family=Manrope:wght@400;500;600;700&display=swap" rel="stylesheet">
  <style>
    :root {
      --ink: #0b0b0a;
      --ink-2: #11110f;
      --panel: #171714;
      --panel-2: #1d1d19;
      --line: rgba(239, 235, 219, .12);
      --line-strong: rgba(239, 235, 219, .22);
      --paper: #efebdb;
      --muted: #9e9a8c;
      --acid: #c8ff32;
      --amber: #ffb257;
      --danger: #ff6f63;
      --cyan: #70d5c7;
      --shadow: 0 28px 70px rgba(0, 0, 0, .36);
      --radius: 18px;
      --mono: "IBM Plex Mono", monospace;
      --sans: "Manrope", sans-serif;
      --serif: "Instrument Serif", serif;
    }

    * { box-sizing: border-box; }
    html { background: var(--ink); }
    body {
      margin: 0;
      min-height: 100vh;
      color: var(--paper);
      font-family: var(--sans);
      background:
        radial-gradient(circle at 78% -10%, rgba(200,255,50,.08), transparent 30rem),
        radial-gradient(circle at -10% 48%, rgba(255,178,87,.055), transparent 28rem),
        var(--ink);
      overflow-x: hidden;
    }

    body::before {
      content: "";
      position: fixed;
      inset: 0;
      pointer-events: none;
      opacity: .16;
      z-index: 100;
      background-image: url("data:image/svg+xml,%3Csvg viewBox='0 0 180 180' xmlns='http://www.w3.org/2000/svg'%3E%3Cfilter id='n'%3E%3CfeTurbulence type='fractalNoise' baseFrequency='.85' numOctaves='4' stitchTiles='stitch'/%3E%3C/filter%3E%3Crect width='100%25' height='100%25' filter='url(%23n)' opacity='.23'/%3E%3C/svg%3E");
      mix-blend-mode: soft-light;
    }

    button, input, textarea, select { font: inherit; }
    button { color: inherit; }
    a { color: inherit; }

    .shell {
      width: min(1540px, calc(100% - 32px));
      margin: 0 auto;
      padding: 20px 0 52px;
      position: relative;
      z-index: 1;
    }

    .topbar {
      display: grid;
      grid-template-columns: 1fr auto 1fr;
      align-items: center;
      min-height: 70px;
      border-bottom: 1px solid var(--line);
      margin-bottom: 18px;
    }

    .brand { display: flex; align-items: baseline; gap: 10px; }
    .brand-mark {
      font-family: var(--serif);
      font-size: 30px;
      letter-spacing: -.03em;
    }
    .brand-suffix {
      color: var(--muted);
      font: 500 10px/1 var(--mono);
      letter-spacing: .16em;
      text-transform: uppercase;
    }

    .top-center {
      display: inline-flex;
      align-items: center;
      gap: 8px;
      border: 1px solid var(--line);
      border-radius: 999px;
      padding: 8px 12px;
      color: var(--muted);
      font: 500 10px/1 var(--mono);
      letter-spacing: .08em;
      text-transform: uppercase;
      background: rgba(255,255,255,.018);
    }
    .pulse {
      width: 7px; height: 7px; border-radius: 50%;
      background: var(--amber);
      box-shadow: 0 0 0 0 rgba(255,178,87,.3);
      animation: pulse 2s infinite;
    }
    .pulse.ready { background: var(--acid); box-shadow: 0 0 18px rgba(200,255,50,.42); }
    .pulse.error { background: var(--danger); }
    @keyframes pulse { 70% { box-shadow: 0 0 0 8px rgba(255,178,87,0); } 100% { box-shadow: 0 0 0 0 rgba(255,178,87,0); } }

    .top-actions { display: flex; justify-content: flex-end; gap: 8px; }
    .mini-btn, .icon-btn {
      border: 1px solid var(--line);
      background: transparent;
      border-radius: 999px;
      min-height: 36px;
      padding: 0 12px;
      cursor: pointer;
      transition: .2s ease;
      font: 500 10px/1 var(--mono);
      letter-spacing: .06em;
      text-transform: uppercase;
    }
    .mini-btn:hover, .icon-btn:hover { border-color: var(--line-strong); background: rgba(255,255,255,.04); transform: translateY(-1px); }

    .studio-grid {
      display: grid;
      grid-template-columns: minmax(300px, .92fr) minmax(500px, 1.55fr) minmax(260px, .78fr);
      gap: 14px;
      align-items: stretch;
    }

    .panel {
      border: 1px solid var(--line);
      border-radius: var(--radius);
      background: linear-gradient(180deg, rgba(255,255,255,.024), rgba(255,255,255,.012));
      box-shadow: inset 0 1px 0 rgba(255,255,255,.025);
      overflow: hidden;
    }

    .panel-head {
      height: 46px;
      display: flex;
      align-items: center;
      justify-content: space-between;
      padding: 0 16px;
      border-bottom: 1px solid var(--line);
      font: 500 10px/1 var(--mono);
      letter-spacing: .13em;
      text-transform: uppercase;
      color: var(--muted);
    }
    .panel-index { color: var(--paper); opacity: .42; }

    .composer-body { padding: 18px; }
    .eyebrow {
      font: 500 10px/1.4 var(--mono);
      letter-spacing: .12em;
      text-transform: uppercase;
      color: var(--muted);
      margin-bottom: 9px;
    }

    .prompt-wrap { position: relative; }
    textarea {
      width: 100%;
      min-height: 164px;
      resize: vertical;
      border: 0;
      outline: 0;
      padding: 0 0 34px;
      color: var(--paper);
      background: transparent;
      font-family: var(--serif);
      font-size: 27px;
      line-height: 1.13;
      letter-spacing: -.015em;
    }
    textarea::placeholder { color: #6f6c62; }
    .charcount {
      position: absolute;
      right: 0; bottom: 8px;
      color: #666359;
      font: 500 9px/1 var(--mono);
    }

    .preset-row {
      display: flex;
      gap: 7px;
      flex-wrap: wrap;
      margin: 4px 0 20px;
    }
    .chip {
      border: 1px solid var(--line);
      background: rgba(255,255,255,.018);
      border-radius: 999px;
      padding: 7px 9px;
      color: #aaa699;
      cursor: pointer;
      transition: .18s ease;
      font: 500 9px/1 var(--mono);
      letter-spacing: .02em;
    }
    .chip:hover { color: var(--ink); background: var(--paper); border-color: var(--paper); transform: translateY(-1px); }

    .control-grid {
      display: grid;
      grid-template-columns: 1fr 1fr;
      gap: 10px;
      padding-top: 16px;
      border-top: 1px solid var(--line);
    }
    .field {
      min-width: 0;
      border: 1px solid var(--line);
      border-radius: 12px;
      padding: 10px 11px;
      background: rgba(0,0,0,.10);
      transition: border-color .18s ease;
    }
    .field:focus-within { border-color: rgba(200,255,50,.45); }
    .field label {
      display: block;
      color: #77746a;
      font: 500 8px/1 var(--mono);
      letter-spacing: .11em;
      text-transform: uppercase;
      margin-bottom: 7px;
    }
    .field input, .field select {
      width: 100%;
      border: 0;
      outline: 0;
      color: var(--paper);
      background: transparent;
      appearance: none;
      font: 500 13px/1 var(--mono);
    }
    select option { background: var(--panel); }

    .advanced {
      margin-top: 10px;
      border: 1px solid var(--line);
      border-radius: 12px;
      overflow: hidden;
    }
    .advanced summary {
      list-style: none;
      cursor: pointer;
      padding: 12px;
      display: flex;
      align-items: center;
      justify-content: space-between;
      color: var(--muted);
      font: 500 9px/1 var(--mono);
      letter-spacing: .08em;
      text-transform: uppercase;
    }
    .advanced summary::-webkit-details-marker { display: none; }
    .advanced-content { padding: 0 12px 12px; display: grid; grid-template-columns: 1fr 1fr; gap: 9px; }

    .generate-btn {
      width: 100%;
      border: 0;
      border-radius: 13px;
      min-height: 56px;
      margin-top: 14px;
      padding: 0 16px;
      display: flex;
      align-items: center;
      justify-content: space-between;
      color: #11120e;
      background: var(--acid);
      cursor: pointer;
      transition: .22s cubic-bezier(.2,.8,.2,1);
      box-shadow: 0 10px 26px rgba(200,255,50,.08);
    }
    .generate-btn:hover { transform: translateY(-2px); box-shadow: 0 14px 35px rgba(200,255,50,.14); }
    .generate-btn:disabled { cursor: not-allowed; opacity: .42; transform: none; }
    .generate-btn span:first-child { font: 700 12px/1 var(--mono); letter-spacing: .04em; text-transform: uppercase; }
    .generate-btn .arrow { font-size: 22px; font-weight: 400; }

    .stable-note {
      display: flex; gap: 9px; align-items: flex-start;
      padding: 12px 2px 0;
      color: #747167;
      font: 400 9px/1.55 var(--mono);
    }
    .stable-note b { color: var(--amber); font-weight: 500; }

    .stage {
      min-height: 670px;
      display: grid;
      grid-template-rows: 46px 1fr;
    }
    .stage-body {
      position: relative;
      min-height: 620px;
      overflow: hidden;
      background:
        linear-gradient(rgba(255,255,255,.022) 1px, transparent 1px),
        linear-gradient(90deg, rgba(255,255,255,.022) 1px, transparent 1px);
      background-size: 32px 32px;
    }
    .stage-body::before {
      content:"";
      position:absolute;
      width: 360px; height: 360px;
      top: 50%; left: 50%;
      transform: translate(-50%, -52%);
      border-radius: 50%;
      background: radial-gradient(circle, rgba(200,255,50,.075), rgba(200,255,50,.015) 42%, transparent 70%);
      filter: blur(1px);
    }

    .record-wrap {
      position: absolute;
      left: 50%;
      top: 45%;
      transform: translate(-50%, -50%);
      width: min(380px, 70%);
      aspect-ratio: 1;
      display: grid;
      place-items: center;
    }
    .record {
      width: 100%; height: 100%;
      border-radius: 50%;
      position: relative;
      background:
        repeating-radial-gradient(circle at center, rgba(239,235,219,.06) 0 1px, transparent 1px 5px),
        radial-gradient(circle at center, #25251f 0 11%, #0b0b0a 11.4% 20%, #20201c 20.5% 21.4%, #0d0d0b 22% 100%);
      border: 1px solid rgba(239,235,219,.16);
      box-shadow: 0 38px 70px rgba(0,0,0,.4), inset 0 0 80px rgba(255,255,255,.025);
      transition: transform .4s ease;
    }
    .record::before {
      content:"";
      position:absolute; inset: 10%;
      border-radius:50%;
      border:1px solid rgba(239,235,219,.06);
      box-shadow: 0 0 0 34px rgba(255,255,255,.008), 0 0 0 72px rgba(255,255,255,.006);
    }
    .record::after {
      content:"ACE";
      position:absolute;
      width: 24%; aspect-ratio:1; border-radius:50%;
      left:38%; top:38%;
      display:grid; place-items:center;
      background: var(--acid); color:#111;
      font: 600 11px/1 var(--mono); letter-spacing:.18em;
      box-shadow: 0 0 35px rgba(200,255,50,.12);
    }
    .record.spinning { animation: spin 4.2s linear infinite; }
    @keyframes spin { to { transform: rotate(360deg); } }

    .orbit {
      position:absolute;
      inset:-24px;
      border: 1px dashed rgba(239,235,219,.12);
      border-radius:50%;
      animation: spin 20s linear infinite reverse;
    }
    .orbit::after {
      content:"";
      width:8px; height:8px; border-radius:50%;
      position:absolute; top: 12%; left: 12%;
      background: var(--acid);
      box-shadow:0 0 18px rgba(200,255,50,.65);
    }

    .stage-status {
      position:absolute;
      left:50%; bottom: 44px;
      transform:translateX(-50%);
      width:min(520px, calc(100% - 48px));
      text-align:center;
    }
    .status-kicker {
      color: var(--acid);
      font: 500 9px/1 var(--mono);
      letter-spacing:.15em;
      text-transform:uppercase;
      margin-bottom:8px;
    }
    .stage-title {
      font-family:var(--serif);
      font-size: 31px;
      line-height:1.05;
      letter-spacing:-.02em;
      margin-bottom:10px;
    }
    .stage-meta { color:#77746a; font:400 9px/1.5 var(--mono); }
    .progress-track {
      height:2px;
      width:100%;
      margin:18px auto 0;
      background: rgba(255,255,255,.08);
      overflow:hidden;
      border-radius: 999px;
    }
    .progress-fill {
      height:100%; width:0%;
      background:var(--acid);
      box-shadow:0 0 12px rgba(200,255,50,.45);
      transition: width .7s ease;
    }

    .result-player {
      position:absolute;
      inset: 18px;
      display:none;
      grid-template-rows:auto 1fr auto;
      border-radius: 14px;
      background: rgba(11,11,10,.94);
      backdrop-filter: blur(14px);
      z-index:5;
    }
    .result-player.show { display:grid; }
    .result-top {
      display:flex; align-items:flex-start; justify-content:space-between;
      padding:22px;
    }
    .result-title {
      max-width:75%;
      font-family:var(--serif);
      font-size:28px;
      line-height:1.06;
    }
    .close-result {
      width:36px; height:36px; border-radius:50%; border:1px solid var(--line);
      background:transparent; cursor:pointer; font-size:20px;
    }
    .wave-area {
      display:grid;
      place-items:center;
      padding: 12px 22px 4px;
    }
    #waveCanvas { width:100%; height:220px; display:block; }
    .player-controls {
      padding:18px 22px 22px;
      border-top:1px solid var(--line);
      display:grid;
      grid-template-columns:auto 1fr auto;
      align-items:center; gap:14px;
    }
    .play-btn {
      width:52px; height:52px; border:0; border-radius:50%;
      background:var(--acid); color:#111; cursor:pointer; font-size:18px;
    }
    .audio-meta { min-width:0; }
    .audio-meta strong {
      display:block;
      white-space:nowrap; overflow:hidden; text-overflow:ellipsis;
      font:500 11px/1.4 var(--mono);
    }
    .audio-meta span { color:var(--muted); font:400 9px/1.4 var(--mono); }
    .download-btn {
      border:1px solid var(--line); border-radius:999px; padding:9px 11px;
      text-decoration:none; font:500 9px/1 var(--mono); text-transform:uppercase;
    }

    .compute-body { padding: 12px; }
    .gpu-card {
      border:1px solid var(--line);
      border-radius:13px;
      padding:13px;
      margin-bottom:10px;
      background: rgba(0,0,0,.12);
      overflow:hidden;
    }
    .gpu-top { display:flex; justify-content:space-between; align-items:center; gap:10px; }
    .gpu-id { font:600 10px/1 var(--mono); letter-spacing:.08em; text-transform:uppercase; }
    .gpu-state {
      border-radius:999px; padding:5px 7px;
      color:var(--muted); background:rgba(255,255,255,.04);
      font:500 8px/1 var(--mono); text-transform:uppercase;
    }
    .gpu-state.ready { color:#12150d; background:var(--acid); }
    .gpu-state.busy { color:#19130b; background:var(--amber); }
    .gpu-state.error { color:#fff; background:var(--danger); }
    .gpu-name {
      margin:12px 0 5px;
      font-family:var(--serif);
      font-size:21px;
    }
    .gpu-detail { color:#716e64; font:400 9px/1.4 var(--mono); min-height:26px; }
    .meter {
      height:4px; border-radius:99px; background:rgba(255,255,255,.07);
      margin:12px 0 7px; overflow:hidden;
    }
    .meter > i { display:block; height:100%; width:0; background:var(--cyan); transition: width .6s ease; }
    .gpu-stats {
      display:flex; justify-content:space-between; gap:8px;
      color:#858176; font:400 8px/1.3 var(--mono);
    }

    .queue-box {
      margin-top:14px; padding-top:14px; border-top:1px solid var(--line);
    }
    .queue-hero {
      display:grid; grid-template-columns:1fr 1fr; gap:8px;
    }
    .stat-tile {
      min-height:82px; border:1px solid var(--line); border-radius:12px;
      padding:11px; background:rgba(255,255,255,.012);
    }
    .stat-tile span { color:#767369; font:500 8px/1 var(--mono); text-transform:uppercase; letter-spacing:.08em; }
    .stat-tile strong { display:block; margin-top:8px; font-family:var(--serif); font-size:28px; font-weight:400; }
    .model-badge {
      margin-top:10px; padding:12px;
      border:1px dashed var(--line-strong); border-radius:12px;
      color:#858176; font:400 8px/1.55 var(--mono);
    }
    .model-badge b { color:var(--paper); font-weight:500; }

    .library {
      margin-top:14px;
      border:1px solid var(--line);
      border-radius:var(--radius);
      overflow:hidden;
      background:rgba(255,255,255,.012);
    }
    .library-head {
      min-height:58px; padding:0 18px;
      display:flex; align-items:center; justify-content:space-between; gap:14px;
      border-bottom:1px solid var(--line);
    }
    .library-head h2 { margin:0; font:400 27px/1 var(--serif); }
    .library-head span { color:var(--muted); font:500 9px/1 var(--mono); text-transform:uppercase; letter-spacing:.09em; }
    .history-grid {
      padding:14px;
      display:grid;
      grid-template-columns:repeat(4, minmax(0,1fr));
      gap:10px;
    }
    .track-card {
      min-width:0;
      border:1px solid var(--line);
      border-radius:14px;
      padding:12px;
      background:linear-gradient(150deg, rgba(255,255,255,.025), rgba(0,0,0,.14));
      transition:.2s ease;
      position:relative;
    }
    .track-card:hover { transform:translateY(-2px); border-color:var(--line-strong); }
    .track-art {
      aspect-ratio: 1.75;
      border-radius:10px;
      margin-bottom:12px;
      position:relative; overflow:hidden;
      background:
        radial-gradient(circle at 28% 35%, hsla(var(--hue),75%,65%,.42), transparent 28%),
        radial-gradient(circle at 72% 60%, hsla(calc(var(--hue) + 70),65%,55%,.24), transparent 34%),
        #151512;
    }
    .track-art::after {
      content:"";
      position:absolute; inset:0;
      background:repeating-linear-gradient(115deg, transparent 0 12px, rgba(255,255,255,.025) 12px 13px);
    }
    .track-art .tiny-record {
      position:absolute; width:74px; height:74px; border-radius:50%;
      right:14px; top:50%; transform:translateY(-50%);
      background:repeating-radial-gradient(circle, #0b0b0a 0 2px, #24241f 2px 4px);
      border:1px solid rgba(255,255,255,.13);
    }
    .track-art .tiny-record::after {
      content:""; position:absolute; width:18px; height:18px; border-radius:50%;
      left:27px; top:27px; background:var(--paper);
    }
    .track-card h3 {
      margin:0 0 7px;
      white-space:nowrap; overflow:hidden; text-overflow:ellipsis;
      font:500 11px/1.35 var(--mono);
    }
    .track-sub { color:#77746a; font:400 8px/1.5 var(--mono); }
    .track-actions { display:flex; gap:6px; margin-top:12px; }
    .track-actions button, .track-actions a {
      flex:1; min-height:31px; border:1px solid var(--line); border-radius:9px;
      background:transparent; text-decoration:none; display:grid; place-items:center;
      cursor:pointer; font:500 8px/1 var(--mono); text-transform:uppercase;
    }
    .track-actions button:hover, .track-actions a:hover { background:var(--paper); color:#111; }

    .empty-library {
      grid-column:1/-1; min-height:150px; display:grid; place-items:center;
      text-align:center; color:#6e6b62; font:400 10px/1.6 var(--mono);
    }

    .toast-stack {
      position:fixed; right:18px; bottom:18px; z-index:70;
      display:grid; gap:8px; width:min(360px, calc(100vw - 36px));
    }
    .toast {
      padding:12px 14px; border:1px solid var(--line-strong); border-radius:12px;
      background:#1b1b17; box-shadow:var(--shadow);
      font:400 10px/1.45 var(--mono);
      animation:toastIn .25s ease both;
    }
    .toast.error { border-color:rgba(255,111,99,.4); }
    @keyframes toastIn { from { opacity:0; transform:translateY(8px); } }


    @media (max-width: 1180px) {
      .studio-grid { grid-template-columns:minmax(300px,.9fr) minmax(480px,1.35fr); }
      .compute { grid-column:1/-1; }
      .compute-body { display:grid; grid-template-columns:1fr 1fr 1fr; gap:10px; }
      .gpu-card { margin:0; }
      .queue-box { margin:0; padding:0; border:0; }
      .history-grid { grid-template-columns:repeat(3,minmax(0,1fr)); }
    }
    @media (max-width: 820px) {
      .shell { width:min(100% - 18px, 720px); padding-top:8px; }
      .topbar { grid-template-columns:1fr auto; }
      .top-center { display:none; }
      .studio-grid { grid-template-columns:1fr; }
      .stage { min-height:610px; }
      .stage-body { min-height:560px; }
      .record-wrap { width:min(330px,72vw); }
      .compute { grid-column:auto; }
      .compute-body { display:block; }
      .gpu-card { margin-bottom:10px; }
      .queue-box { margin-top:14px; padding-top:14px; border-top:1px solid var(--line); }
      .history-grid { grid-template-columns:repeat(2,minmax(0,1fr)); }
    }
    @media (max-width: 540px) {
      .brand-mark { font-size:25px; }
      .brand-suffix { display:none; }
      .control-grid { grid-template-columns:1fr 1fr; }
      textarea { font-size:25px; }
      .history-grid { grid-template-columns:1fr; }
      .result-top { padding:16px; }
      .result-title { font-size:24px; }
      #waveCanvas { height:160px; }
    }
    @media (prefers-reduced-motion: reduce) {
      *, *::before, *::after { animation-duration:.001ms !important; animation-iteration-count:1 !important; scroll-behavior:auto !important; }
    }
  </style>
</head>
<body>
  <main class="shell">
    <header class="topbar">
      <div class="brand">
        <div class="brand-mark">ACE Studio</div>
        <div class="brand-suffix">Generative Music Console</div>
      </div>
      <div class="top-center">
        <i class="pulse" id="systemPulse"></i>
        <span id="systemText">Connecting workers</span>
      </div>
      <div class="top-actions">
        <button class="mini-btn" id="refreshBtn">Refresh</button>
      </div>
    </header>

    <section class="studio-grid">
      <aside class="panel composer">
        <div class="panel-head">
          <span>Composer</span><span class="panel-index">01</span>
        </div>
        <div class="composer-body">
          <div class="eyebrow">Describe the record</div>
          <div class="prompt-wrap">
            <textarea id="prompt" maxlength="700" placeholder="A solitary piano remembers the last train home…">solo grand piano, emotional Japanese cinematic instrumental, nostalgic summer evening, delicate melody, expressive dynamics, intimate room tone, no vocals, no drums</textarea>
            <div class="charcount"><span id="chars">0</span>/700</div>
          </div>

          <div class="preset-row">
            <button class="chip" data-preset="cinema">Cinematic Piano</button>
            <button class="chip" data-preset="lofi">Lo-fi Night</button>
            <button class="chip" data-preset="neo">Neo-classical</button>
            <button class="chip" data-preset="ambient">Ambient Score</button>
          </div>

          <div class="control-grid">
            <div class="field">
              <label>Duration</label>
              <select id="duration">
                <option value="20">20 sec</option>
                <option value="30" selected>30 sec</option>
                <option value="45">45 sec</option>
                <option value="60">60 sec</option>
                <option value="90">90 sec</option>
              </select>
            </div>
            <div class="field">
              <label>BPM</label>
              <input id="bpm" type="number" min="30" max="300" value="82" />
            </div>
            <div class="field">
              <label>Key</label>
              <select id="keyscale">
                <option>A minor</option><option>C Major</option><option>D Major</option>
                <option>E minor</option><option>F Major</option><option>G Major</option>
                <option>B minor</option><option>D minor</option>
              </select>
            </div>
            <div class="field">
              <label>Time</label>
              <select id="timesignature">
                <option value="4">4 / 4</option>
                <option value="3">3 / 4</option>
                <option value="6">6 / 8</option>
                <option value="2">2 / 4</option>
              </select>
            </div>
          </div>

          <details class="advanced">
            <summary><span>Advanced render</span><span>＋</span></summary>
            <div class="advanced-content">
              <div class="field">
                <label>Seed</label>
                <input id="seed" type="number" min="0" placeholder="Random" />
              </div>
              <div class="field">
                <label>Turbo steps</label>
                <select id="steps">
                  <option value="8" selected>8 · Fast</option>
                  <option value="12">12</option>
                  <option value="16">16</option>
                </select>
              </div>
            </div>
          </details>

          <button class="generate-btn" id="generateBtn">
            <span>Render on dual T4</span><span class="arrow">↗</span>
          </button>

          <div class="stable-note">
            <span>●</span>
            <span><b>T4 stable profile.</b> Turbo DiT, instrumental, batch 1, no 5Hz LM. Requests are automatically routed to the shorter GPU queue.</span>
          </div>
        </div>
      </aside>

      <section class="panel stage">
        <div class="panel-head">
          <span>Master / Preview</span><span class="panel-index">02</span>
        </div>
        <div class="stage-body">
          <div class="record-wrap">
            <div class="orbit"></div>
            <div class="record" id="record"></div>
          </div>

          <div class="stage-status">
            <div class="status-kicker" id="statusKicker">Ready for direction</div>
            <div class="stage-title" id="stageTitle">Write a scene. Render a record.</div>
            <div class="stage-meta" id="stageMeta">ACE-Step 1.5 Turbo · dual NVIDIA T4 · instrumental pipeline</div>
            <div class="progress-track"><div class="progress-fill" id="progressFill"></div></div>
          </div>

          <div class="result-player" id="resultPlayer">
            <div class="result-top">
              <div>
                <div class="eyebrow">Latest master</div>
                <div class="result-title" id="resultTitle"></div>
              </div>
              <button class="close-result" id="closeResult" aria-label="Close player">×</button>
            </div>
            <div class="wave-area"><canvas id="waveCanvas"></canvas></div>
            <div class="player-controls">
              <button class="play-btn" id="playBtn" aria-label="Play or pause">▶</button>
              <div class="audio-meta">
                <strong id="audioName">ACE render</strong>
                <span id="audioInfo">—</span>
              </div>
              <a class="download-btn" id="downloadBtn" href="#" download>Download WAV</a>
            </div>
          </div>
          <audio id="audio" preload="metadata"></audio>
        </div>
      </section>

      <aside class="panel compute">
        <div class="panel-head">
          <span>Compute</span><span class="panel-index">03</span>
        </div>
        <div class="compute-body">
          <div id="gpuCards"></div>
          <div class="queue-box">
            <div class="queue-hero">
              <div class="stat-tile"><span>Active</span><strong id="activeJobs">0</strong></div>
              <div class="stat-tile"><span>Masters</span><strong id="doneJobs">0</strong></div>
            </div>
            <div class="model-badge">
              <b id="modelName">acestep-v15-turbo</b><br>
              DiT-only · no LM reasoning<br>
              Same-origin API · polling via Tunnel
            </div>
          </div>
        </div>
      </aside>
    </section>

    <section class="library">
      <div class="library-head">
        <h2>Session Masters</h2>
        <span id="libraryCount">0 renders</span>
      </div>
      <div class="history-grid" id="historyGrid">
        <div class="empty-library">No masters yet.<br>Render your first 30-second study above.</div>
      </div>
    </section>
  </main>

  <div class="toast-stack" id="toasts"></div>

  <script>
    const $ = (s) => document.querySelector(s);
    const state = { currentJob: null, currentAudioJob: null, statusTimer: null, historySignature: "" };

    const presets = {
      cinema: "solo grand piano, emotional Japanese cinematic instrumental, nostalgic summer evening, delicate melody, expressive dynamics, intimate room tone, no vocals, no drums",
      lofi: "dusty lo-fi instrumental, warm upright piano, soft tape hiss, late-night city rain, restrained drums, mellow bass, intimate and nostalgic, no vocals",
      neo: "neo-classical piano instrumental, repeating minimalist motif, felt piano, gradual harmonic development, fragile dynamics, spacious concert hall, no vocals",
      ambient: "ambient cinematic instrumental, sparse piano notes, soft evolving pads, distant texture, slow luminous harmony, meditative night atmosphere, no vocals"
    };

    function toast(message, type="") {
      const el = document.createElement("div");
      el.className = `toast ${type}`;
      el.textContent = message;
      $("#toasts").appendChild(el);
      setTimeout(() => el.remove(), 4200);
    }

    async function api(path, options={}) {
      options.headers = Object.assign({}, options.headers || {}, {
        "Content-Type": "application/json"
      });
      const res = await fetch(path, options);
      let body = null;
      try { body = await res.json(); } catch {}
      if (!res.ok) throw new Error(body?.detail || `HTTP ${res.status}`);
      return body;
    }

    function esc(s="") {
      return String(s).replace(/[&<>"']/g, c => ({"&":"&amp;","<":"&lt;",">":"&gt;","\"":"&quot;","'":"&#039;"}[c]));
    }

    function shortPrompt(prompt, n=62) {
      return prompt.length > n ? prompt.slice(0,n).trim() + "…" : prompt;
    }

    function formatAge(ts) {
      if (!ts) return "—";
      const d = Math.max(0, Date.now()/1000 - ts);
      if (d < 60) return `${Math.round(d)}s ago`;
      if (d < 3600) return `${Math.round(d/60)}m ago`;
      return `${Math.round(d/3600)}h ago`;
    }

    function updateCharCount() { $("#chars").textContent = $("#prompt").value.length; }

    function renderGPU(g) {
      const t = g.telemetry || {};
      const pct = t.memory_total_mb ? Math.round((t.memory_used_mb / t.memory_total_mb) * 100) : 0;
      return `
        <div class="gpu-card">
          <div class="gpu-top">
            <span class="gpu-id">GPU ${g.id ?? "?"}</span>
            <span class="gpu-state ${esc(g.state || "")}">${esc(g.state || "booting")}</span>
          </div>
          <div class="gpu-name">${esc(g.name || "NVIDIA T4")}</div>
          <div class="gpu-detail">${esc(g.detail || "Worker booting…")} · queue ${g.queue ?? 0}</div>
          <div class="meter"><i style="width:${pct}%"></i></div>
          <div class="gpu-stats">
            <span>${t.memory_used_mb != null ? Math.round(t.memory_used_mb/1024*10)/10 + " / " + Math.round(t.memory_total_mb/1024*10)/10 + " GB" : "VRAM —"}</span>
            <span>${t.utilization != null ? t.utilization + "% · " + t.temperature + "°C" : "UTIL —"}</span>
          </div>
        </div>`;
    }

    async function refreshStatus() {
      try {
        const s = await api("/api/status");
        $("#gpuCards").innerHTML = s.gpus.map(renderGPU).join("");
        $("#activeJobs").textContent = s.jobs_active;
        $("#doneJobs").textContent = s.jobs_done;
        $("#modelName").textContent = s.model;

        const bad = s.gpus.some(g => g.state === "error");
        const ready = s.gpus.length && s.gpus.every(g => ["ready","busy"].includes(g.state));
        $("#systemPulse").className = "pulse " + (bad ? "error" : ready ? "ready" : "");
        $("#systemText").textContent = bad ? "Worker error" : ready ? "Dual workers online" : "Loading models";

        $("#generateBtn").disabled = s.gpus.length === 0 || s.gpus.every(g => g.state === "error");

        if (state.currentJob) {
          try {
            const job = await api(`/api/jobs/${state.currentJob}`);
            updateCurrentJob(job);
          } catch {}
        }

        if (Math.random() < .45) refreshHistory();
      } catch (err) {
        $("#systemText").textContent = "Server offline";
        $("#systemPulse").className = "pulse error";
      }
    }

    function updateCurrentJob(job) {
      const p = Math.round((job.progress || 0) * 100);
      $("#progressFill").style.width = `${p}%`;

      if (job.status === "queued") {
        $("#record").classList.add("spinning");
        $("#statusKicker").textContent = `Queued · GPU ${job.gpu}`;
        $("#stageTitle").textContent = "Waiting for the next open deck.";
        $("#stageMeta").textContent = `${job.duration}s · ${job.bpm} BPM · ${job.keyscale} · ${p}%`;
      } else if (job.status === "running") {
        $("#record").classList.add("spinning");
        $("#statusKicker").textContent = `Rendering · GPU ${job.gpu}`;
        $("#stageTitle").textContent = job.stage || "Diffusing the next phrase…";
        $("#stageMeta").textContent = `${job.duration}s · ${job.bpm} BPM · ${job.keyscale} · ${p}%`;
      } else if (job.status === "done") {
        $("#record").classList.remove("spinning");
        $("#statusKicker").textContent = `Mastered · ${job.elapsed_seconds || "—"}s`;
        $("#stageTitle").textContent = "Your record is ready.";
        $("#stageMeta").textContent = `GPU ${job.gpu} · seed ${job.seed} · ${job.duration}s`;
        $("#progressFill").style.width = "100%";
        if (state.currentAudioJob !== job.id) {
          openResult(job);
          refreshHistory(true);
        }
      } else if (job.status === "error") {
        $("#record").classList.remove("spinning");
        $("#statusKicker").textContent = "Render failed";
        $("#stageTitle").textContent = "The take collapsed.";
        $("#stageMeta").textContent = job.error || "See Kaggle server log.";
        $("#progressFill").style.width = "0%";
        toast(job.error || "Generation failed", "error");
        state.currentJob = null;
      }
    }

    async function generate() {
      const prompt = $("#prompt").value.trim();
      if (prompt.length < 3) return toast("Write a prompt first.", "error");

      const payload = {
        prompt,
        duration: Number($("#duration").value),
        bpm: Number($("#bpm").value),
        keyscale: $("#keyscale").value,
        timesignature: $("#timesignature").value,
        steps: Number($("#steps").value),
        seed: $("#seed").value ? Number($("#seed").value) : null
      };

      try {
        $("#generateBtn").disabled = true;
        const job = await api("/api/generate", { method:"POST", body:JSON.stringify(payload) });
        state.currentJob = job.id;
        state.currentAudioJob = null;
        $("#resultPlayer").classList.remove("show");
        $("#progressFill").style.width = "3%";
        $("#record").classList.add("spinning");
        $("#statusKicker").textContent = `Queued · GPU ${job.gpu}`;
        $("#stageTitle").textContent = "The deck is warming up.";
        $("#stageMeta").textContent = `${payload.duration}s · ${payload.bpm} BPM · ${payload.keyscale}`;
        toast(`Queued on GPU ${job.gpu}`);
      } catch (err) {
        toast(err.message, "error");
      } finally {
        setTimeout(() => $("#generateBtn").disabled = false, 900);
      }
    }

    async function drawWave(url) {
      const canvas = $("#waveCanvas");
      const ctx = canvas.getContext("2d");
      const rect = canvas.getBoundingClientRect();
      const dpr = Math.min(window.devicePixelRatio || 1, 2);
      canvas.width = Math.max(1, Math.floor(rect.width * dpr));
      canvas.height = Math.max(1, Math.floor(rect.height * dpr));
      ctx.scale(dpr, dpr);
      ctx.clearRect(0,0,rect.width,rect.height);

      try {
        const arr = await fetch(url).then(r => r.arrayBuffer());
        const AC = window.AudioContext || window.webkitAudioContext;
        const ac = new AC();
        const buf = await ac.decodeAudioData(arr.slice(0));
        const data = buf.getChannelData(0);
        const bars = Math.max(80, Math.floor(rect.width / 4));
        const step = Math.max(1, Math.floor(data.length / bars));
        const mid = rect.height / 2;

        ctx.lineWidth = 1;
        for (let i=0;i<bars;i++) {
          let peak = 0;
          const start = i*step;
          for (let j=0;j<step;j+=Math.max(1,Math.floor(step/24))) {
            peak = Math.max(peak, Math.abs(data[start+j] || 0));
          }
          const x = (i/(bars-1))*rect.width;
          const h = Math.max(2, peak * rect.height * .78);
          ctx.strokeStyle = i/bars < .68 ? "rgba(200,255,50,.78)" : "rgba(239,235,219,.25)";
          ctx.beginPath();
          ctx.moveTo(x, mid-h/2);
          ctx.lineTo(x, mid+h/2);
          ctx.stroke();
        }
        await ac.close();
      } catch {
        ctx.strokeStyle = "rgba(239,235,219,.18)";
        for(let x=0;x<rect.width;x+=6){
          const h = 8 + Math.sin(x*.08)*6;
          ctx.beginPath(); ctx.moveTo(x,rect.height/2-h); ctx.lineTo(x,rect.height/2+h); ctx.stroke();
        }
      }
    }

    function openResult(job) {
      if (!job.audio_url) return;
      state.currentAudioJob = job.id;
      const audioUrl = job.audio_url;
      const downloadUrl = job.download_url;
      $("#resultTitle").textContent = shortPrompt(job.prompt, 88);
      $("#audioName").textContent = `ACE · ${job.id.slice(0,8)}`;
      $("#audioInfo").textContent = `${job.duration}s · ${job.bpm} BPM · ${job.keyscale} · seed ${job.seed}`;
      $("#audio").src = audioUrl;
      $("#downloadBtn").href = downloadUrl;
      $("#resultPlayer").classList.add("show");
      $("#playBtn").textContent = "▶";
      drawWave(audioUrl);
    }

    function historyCard(j) {
      const hue = parseInt(j.id.slice(0,4),16) % 360;
      const status = j.status === "done" ? `${j.duration}s · GPU ${j.gpu}` : `${j.status} · ${formatAge(j.created_at)}`;
      return `
        <article class="track-card">
          <div class="track-art" style="--hue:${hue}">
            <div class="tiny-record"></div>
          </div>
          <h3 title="${esc(j.prompt)}">${esc(shortPrompt(j.prompt,46))}</h3>
          <div class="track-sub">${esc(status)} · ${esc(j.keyscale || "")}</div>
          <div class="track-actions">
            ${j.status === "done" ? `<button data-play="${j.id}">Play</button><a href="${j.download_url}">WAV</a>` : `<button disabled>${esc(j.stage || j.status)}</button>`}
          </div>
        </article>`;
    }

    async function refreshHistory(force=false) {
      try {
        const jobs = await api("/api/history?limit=32");
        const sig = jobs.map(j => `${j.id}:${j.status}:${j.updated_at}`).join("|");
        if (!force && sig === state.historySignature) return;
        state.historySignature = sig;
        $("#libraryCount").textContent = `${jobs.length} render${jobs.length===1?"":"s"}`;
        $("#historyGrid").innerHTML = jobs.length
          ? jobs.map(historyCard).join("")
          : `<div class="empty-library">No masters yet.<br>Render your first 30-second study above.</div>`;

        document.querySelectorAll("[data-play]").forEach(btn => {
          btn.addEventListener("click", async () => {
            try {
              const job = await api(`/api/jobs/${btn.dataset.play}`);
              openResult(job);
              window.scrollTo({top:0, behavior:"smooth"});
            } catch(err) { toast(err.message,"error"); }
          });
        });
      } catch {}
    }

    $("#generateBtn").addEventListener("click", generate);
    $("#refreshBtn").addEventListener("click", () => { refreshStatus(); refreshHistory(true); });
    $("#prompt").addEventListener("input", updateCharCount);
    document.querySelectorAll("[data-preset]").forEach(btn => btn.addEventListener("click", () => {
      $("#prompt").value = presets[btn.dataset.preset];
      updateCharCount();
    }));

    $("#closeResult").addEventListener("click", () => $("#resultPlayer").classList.remove("show"));
    $("#playBtn").addEventListener("click", () => {
      const audio = $("#audio");
      if (audio.paused) audio.play(); else audio.pause();
    });
    $("#audio").addEventListener("play", () => $("#playBtn").textContent = "❚❚");
    $("#audio").addEventListener("pause", () => $("#playBtn").textContent = "▶");
    $("#audio").addEventListener("ended", () => $("#playBtn").textContent = "▶");

    async function boot() {
      updateCharCount();
      await refreshStatus();
      await refreshHistory(true);
      state.statusTimer = setInterval(refreshStatus, 1800);
    }

    boot();
  </script>
</body>
</html>
"""

SERVER_FILE.write_text(server_source, encoding="utf-8")
UI_FILE.write_text(ui_source, encoding="utf-8")

print("Saved:", SERVER_FILE)
print("Saved:", UI_FILE)
print("Server bytes:", SERVER_FILE.stat().st_size)
print("UI bytes:", UI_FILE.stat().st_size)

Saved: /kaggle/working/ACE-Step-1.5/ace_studio_server.py
Saved: /kaggle/working/ACE-Step-1.5/ace_studio_ui.html
Server bytes: 16491
UI bytes: 40832


## Studio UI 信息架构

```text
┌──────────────────────────────────────────────────────────────────────┐
│ ACE STUDIO              SYSTEM / MODEL STATUS             SESSION   │
├────────────────┬────────────────────────────────┬────────────────────┤
│ COMPOSER       │ MASTER / NOW RENDERING         │ COMPUTE            │
│                │                                │                    │
│ Prompt         │ animated vinyl / stage         │ GPU 0 · VRAM       │
│ Presets        │ live progress                  │ GPU 1 · VRAM       │
│ Duration       │ result player                  │ utilization / temp │
│ BPM / Key      │ canvas waveform                │ queue telemetry    │
│ Time Signature │ metadata                       │                    │
│ Seed / Steps   │                                │                    │
│ GENERATE       │                                │                    │
├────────────────┴────────────────────────────────┴────────────────────┤
│ SESSION MASTERS · generation history · replay · WAV download        │
└──────────────────────────────────────────────────────────────────────┘
```

### 运行策略

- 一个 FastAPI 主进程负责 UI / API / 调度。
- 两个独立 GPU Worker 启动后各自只看到一张 T4。
- 模型只加载一次；生成结束后 Worker 回到 ready，下一首直接复用热模型。
- 请求自动分配给队列较短的 GPU。
- 页面每约 1.8 秒拉取一次状态；CF Quick Tunnel 不需要支持 SSE。
- WAV 和 API 同源，不存在浏览器 CORS / Private Network Request 问题。

In [5]:
# Cell 4 — 启动 ACE Studio（FastAPI + 两个常驻 T4 Worker）
#
# 修复：
# 1. 自动补回 FastAPI Query import
# 2. 无密码 / 无 Studio Key
# 3. 清理旧服务
# 4. 保留已有模型和历史结果
# 5. 隔离 Kaggle Python 环境污染

from pathlib import Path
import os
import signal
import subprocess
import time
import urllib.request

PROJECT = Path("/kaggle/working/ACE-Step-1.5")
ACE_PY = PROJECT / ".venv" / "bin" / "python"
SERVER_FILE = PROJECT / "ace_studio_server.py"

SERVER_PID = Path("/kaggle/working/ace_studio_server.pid")
SERVER_LOG = Path("/kaggle/working/ace_studio_server.log")

PORT = 7860


# ---------------------------------------------------------
# 1. 停止旧服务
# ---------------------------------------------------------

def stop_pid_group(pidfile: Path):
    if not pidfile.exists():
        return

    try:
        pid = int(pidfile.read_text().strip())

        try:
            os.killpg(pid, signal.SIGTERM)
        except ProcessLookupError:
            pass

        print("Stopping previous process group:", pid)
        time.sleep(2)

    except Exception as e:
        print("Previous-process cleanup:", repr(e))

    finally:
        pidfile.unlink(missing_ok=True)


stop_pid_group(SERVER_PID)


# ---------------------------------------------------------
# 2. 自动修复 ace_studio_server.py
# ---------------------------------------------------------

assert SERVER_FILE.exists(), (
    f"{SERVER_FILE} 不存在，请先运行生成 Studio 服务文件的 Cell。"
)

source = SERVER_FILE.read_text(encoding="utf-8")

# 上一版无密码修改时误删了 Query import。
if "Query(" in source:

    if "from fastapi import FastAPI, HTTPException\n" in source:
        source = source.replace(
            "from fastapi import FastAPI, HTTPException\n",
            "from fastapi import FastAPI, HTTPException, Query\n",
            1,
        )

    elif "from fastapi import FastAPI, HTTPException\r\n" in source:
        source = source.replace(
            "from fastapi import FastAPI, HTTPException\r\n",
            "from fastapi import FastAPI, HTTPException, Query\r\n",
            1,
        )

    elif "Query" not in source.split("\n", 40)[0:40]:
        raise RuntimeError(
            "发现代码使用 Query()，但无法自动定位 FastAPI import。"
        )

SERVER_FILE.write_text(source, encoding="utf-8")

print("✓ FastAPI Query import checked/fixed")


# ---------------------------------------------------------
# 3. 清掉旧认证遗留
# ---------------------------------------------------------

Path(
    "/kaggle/working/ace_studio_access_key.txt"
).unlink(missing_ok=True)


# ---------------------------------------------------------
# 4. 干净的 ACE-Step 子进程环境
# ---------------------------------------------------------

def clean_server_env():
    env = os.environ.copy()

    # 防止 Kaggle kernel 环境污染 ACE-Step venv
    env.pop("PYTHONPATH", None)
    env.pop("PYTHONHOME", None)

    env["PYTHONNOUSERSITE"] = "1"

    # 禁止继承 notebook matplotlib backend
    env["MPLBACKEND"] = "Agg"
    env["MPLCONFIGDIR"] = "/tmp/ace_studio_matplotlib"

    env["PYTHONUNBUFFERED"] = "1"

    # Studio 配置
    env["ACE_STUDIO_GPUS"] = "0,1"
    env["ACE_STUDIO_MODEL"] = "acestep-v15-turbo"
    env["ACE_STUDIO_PORT"] = str(PORT)
    env["ACE_STUDIO_OUTPUTS"] = (
        "/kaggle/working/ace_studio_outputs"
    )

    # 明确删除旧认证变量
    env.pop("ACE_STUDIO_KEY", None)

    return env


# ---------------------------------------------------------
# 5. 启动 FastAPI
# ---------------------------------------------------------

log_handle = open(
    SERVER_LOG,
    "w",
    encoding="utf-8",
    buffering=1,
)

server_proc = subprocess.Popen(
    [
        str(ACE_PY),
        str(SERVER_FILE),
    ],
    cwd=PROJECT,
    env=clean_server_env(),
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    text=True,
    start_new_session=True,
)

log_handle.close()

SERVER_PID.write_text(str(server_proc.pid))

print("ACE Studio PID:", server_proc.pid)


# ---------------------------------------------------------
# 6. 等待 HTTP 服务启动
# ---------------------------------------------------------

local_url = f"http://127.0.0.1:{PORT}/"

deadline = time.time() + 90
last_error = None

while time.time() < deadline:

    if server_proc.poll() is not None:
        log_text = (
            SERVER_LOG.read_text(errors="replace")
            if SERVER_LOG.exists()
            else ""
        )

        raise RuntimeError(
            "ACE Studio 进程提前退出。\n\n"
            + log_text[-10000:]
        )

    try:
        with urllib.request.urlopen(
            local_url,
            timeout=2,
        ) as r:

            if r.status == 200:
                break

    except Exception as e:
        last_error = e
        time.sleep(1)

else:
    log_text = (
        SERVER_LOG.read_text(errors="replace")
        if SERVER_LOG.exists()
        else ""
    )

    raise RuntimeError(
        f"ACE Studio 没有在 {PORT} 端口启动。\n"
        f"Last error: {last_error}\n\n"
        + log_text[-10000:]
    )


# ---------------------------------------------------------
# 7. 检查 API
# ---------------------------------------------------------

status_url = (
    f"http://127.0.0.1:{PORT}/api/status"
)

try:
    with urllib.request.urlopen(
        status_url,
        timeout=5,
    ) as r:
        print("✓ /api/status:", r.status)

except Exception as e:
    print("⚠ /api/status check:", repr(e))


print()
print("✓ ACE Studio HTTP server ready")
print("Local:", local_url)
print("Authentication: OFF")
print()
print(
    "两张 T4 会继续在后台加载 ACE-Step。"
)
print(
    "页面显示 Loading → Ready 即代表模型加载完成。"
)

Stopping previous process group: 276
✓ FastAPI Query import checked/fixed
ACE Studio PID: 282
✓ /api/status: 200

✓ ACE Studio HTTP server ready
Local: http://127.0.0.1:7860/
Authentication: OFF

两张 T4 会继续在后台加载 ACE-Step。
页面显示 Loading → Ready 即代表模型加载完成。


In [6]:
# Cell 5 — 启动 Cloudflare Quick Tunnel（无密码）
#
# 输出：
#   https://xxxxxxxx.trycloudflare.com
#
# 直接打开即可使用，不需要 ?key=、密码或请求 Header。

from pathlib import Path
from IPython.display import HTML, display
import os
import re
import signal
import stat
import subprocess
import time
import urllib.request

CLOUDFLARED = Path("/kaggle/working/cloudflared")
TUNNEL_PID = Path("/kaggle/working/ace_studio_tunnel.pid")
TUNNEL_LOG = Path("/kaggle/working/ace_studio_tunnel.log")

stop_pid_group(TUNNEL_PID)

if not CLOUDFLARED.exists():
    print("Downloading cloudflared…")
    subprocess.run(
        [
            "curl", "-L", "--fail", "--silent", "--show-error",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "-o", str(CLOUDFLARED),
        ],
        check=True,
    )
    CLOUDFLARED.chmod(CLOUDFLARED.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

print(subprocess.check_output([str(CLOUDFLARED), "--version"], text=True).strip())

tunnel_log = open(TUNNEL_LOG, "w", encoding="utf-8", buffering=1)
tunnel_proc = subprocess.Popen(
    [
        str(CLOUDFLARED),
        "tunnel",
        "--url", "http://127.0.0.1:7860",
    ],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
    text=True,
    start_new_session=True,
)
tunnel_log.close()
TUNNEL_PID.write_text(str(tunnel_proc.pid))

pattern = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
deadline = time.time() + 75
PUBLIC_URL = None

while time.time() < deadline:
    if tunnel_proc.poll() is not None:
        raise RuntimeError(
            "cloudflared 提前退出。\n\n" +
            TUNNEL_LOG.read_text(errors="replace")[-8000:]
        )

    text = TUNNEL_LOG.read_text(errors="replace") if TUNNEL_LOG.exists() else ""
    match = pattern.search(text)
    if match:
        PUBLIC_URL = match.group(0)
        break
    time.sleep(1)

if not PUBLIC_URL:
    raise RuntimeError(
        "75 秒内没有拿到 trycloudflare URL。\n\n" +
        TUNNEL_LOG.read_text(errors="replace")[-8000:]
    )

# 验证公网入口。
for _ in range(20):
    try:
        req = urllib.request.Request(
            PUBLIC_URL + "/",
            headers={"User-Agent": "ACE-Studio-Kaggle"},
        )
        with urllib.request.urlopen(req, timeout=5) as r:
            if r.status == 200:
                break
    except Exception:
        time.sleep(1)

print("\nACE STUDIO PUBLIC URL")
print(PUBLIC_URL)
print("\nAuthentication: OFF — open the URL directly.")

display(HTML(f"""
<div style="
  margin:18px 0;padding:20px 22px;border:1px solid #30302b;border-radius:18px;
  background:#0b0b0a;color:#efebdb;font-family:ui-monospace,monospace;
">
  <div style="font-size:11px;letter-spacing:.14em;color:#9e9a8c;margin-bottom:10px">
    ACE STUDIO · PUBLIC CLOUDFLARE TUNNEL
  </div>
  <a href="{PUBLIC_URL}" target="_blank" style="
    display:inline-block;padding:13px 18px;border-radius:999px;
    background:#c8ff32;color:#0b0b0a;text-decoration:none;font-weight:700;
  ">OPEN MUSIC STUDIO ↗</a>
  <div style="font-size:11px;color:#77746a;margin-top:12px">
    No password. Anyone with this URL can use the current Kaggle GPU session.
  </div>
</div>
"""))


cloudflared version 2026.8.3 (built 2026-08-31-10:04 UTC)

ACE STUDIO PUBLIC URL
https://retreat-montreal-another-expressed.trycloudflare.com

Authentication: OFF — open the URL directly.


In [ ]:
# Cell 6 — 运维 / 状态检查
#
# 平时无需运行。出问题时运行：
# - 两张 GPU
# - Studio API 状态
# - server log tail
# - tunnel log tail

from pathlib import Path
import json
import subprocess
import urllib.request

print("=== NVIDIA ===")
subprocess.run(["nvidia-smi"], check=False)

print("\n=== STUDIO API ===")
try:
    with urllib.request.urlopen("http://127.0.0.1:7860/api/status", timeout=5) as r:
        print(json.dumps(json.load(r), ensure_ascii=False, indent=2))
except Exception as e:
    print("API status error:", repr(e))

for name, path in [
    ("SERVER LOG", Path("/kaggle/working/ace_studio_server.log")),
    ("TUNNEL LOG", Path("/kaggle/working/ace_studio_tunnel.log")),
]:
    print(f"\n=== {name} (tail) ===")
    if path.exists():
        print(path.read_text(errors="replace")[-7000:])
    else:
        print("not found")


## 使用方式

完成到 **Cell 5** 后，点击 `OPEN MUSIC STUDIO ↗`。

页面里可以直接：

- 输入音乐 Prompt；
- 使用 Cinematic Piano / Lo-fi Night / Neo-classical / Ambient Score 预设；
- 调整 Duration、BPM、Key、Time Signature；
- Advanced 中设置 Seed 和 Turbo Steps；
- 查看两张 T4 的 **Ready / Busy / Loading、VRAM、GPU Util、Temperature、Queue**；
- 查看生成进度；
- 播放生成 WAV、查看 waveform；
- 在 `Session Masters` 中回放和下载历史作品。

### 这版删掉了什么

旧 Notebook 中以下流程已经不再需要：

```text
手工写 one-shot worker
→ 单卡 smoke test
→ Notebook Audio 播放
→ 再起两个一次性进程
→ 再手工看日志
```

现在统一为：

```text
Setup
  ↓
ACE Studio Server
  ↓
GPU0 Worker ─┐
GPU1 Worker ─┴─ persistent queue
  ↓
Custom UI
  ↓
Cloudflare HTTPS
```

Cloudflare Quick Tunnel 适合作为 Kaggle 临时会话入口；Notebook 停止后地址也随之失效。需要长期固定域名时，再把最后一格换成 **Named Tunnel** 即可。